# 05 - Real-Time Structured Streaming Simulation
This notebook simulates real-time transaction streams and performs micro-batch calculations for revenue dashboards and alert rules.

In [ ]:
# --- SELF-CONTAINED SETUP AND DATA INGESTION ---
# This cell handles imports, SparkSession initialization, and data preparation
# so this notebook can run independently of previous notebooks.

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum, avg, when, isnan, round, desc, asc, year, month, to_date, countDistinct, date_sub, abs, max, datediff, lit

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("BigDataMarketing_Submodule") \
    .getOrCreate()

# Find dataset
file_path = "../data/online_retail_II.xlsx"
if not os.path.exists(file_path):
    file_path = "online_retail_II.xlsx"
if not os.path.exists(file_path):
    file_path = "data/online_retail_II.xlsx"
if not os.path.exists(file_path):
    file_path = "/teamspace/studios/this_studio/Datasetes/online_retail_II.xlsx"

print(f"Loading dataset from: {file_path}")

# Ingest raw data
pandas_df_1 = pd.read_excel(file_path, sheet_name="Year 2009-2010", engine="openpyxl")
pandas_df_2 = pd.read_excel(file_path, sheet_name="Year 2010-2011", engine="openpyxl")
pandas_df = pd.concat([pandas_df_1, pandas_df_2], ignore_index=True)

# Clean and prepare DataFrame
retail_df = spark.createDataFrame(pandas_df)
retail_df = retail_df.withColumnRenamed("Customer ID", "CustomerID")
retail_df = retail_df.filter(col("CustomerID").isNotNull() & ~isnan(col("CustomerID")))
retail_df = retail_df.withColumn("CustomerID", col("CustomerID").cast("integer").cast("string"))
retail_df = retail_df.distinct()

# Remove cancellations and invalid records for core sales analysis
retail_df = retail_df.filter(
    (~col("Invoice").startswith("C")) &
    (col("Quantity") > 0) &
    (col("Price") > 0)
).cache()

print(f"Data preparation complete. Clean rows: {retail_df.count():,}")


In [68]:
import os
import time

stream_input_dir  = "/tmp/retail_stream_input"
stream_output_dir = "/tmp/retail_stream_output"

os.makedirs(stream_input_dir,  exist_ok=True)
os.makedirs(stream_output_dir, exist_ok=True)

# Split into 5 batches — simulates transactions arriving

batches = retail_df.randomSplit([0.2, 0.2, 0.2, 0.2, 0.2], seed=42)

print("=== STREAMING SETUP ===")
print(f"Input directory:  {stream_input_dir}")
print(f"Output directory: {stream_output_dir}")
print(f"Total batches:    {len(batches)}")
for i, b in enumerate(batches):
    print(f"  Batch {i+1}: {b.count():,} transactions")

=== STREAMING SETUP ===
Input directory:  /tmp/retail_stream_input
Output directory: /tmp/retail_stream_output
Total batches:    5


  Batch 1: 156,033 transactions


  Batch 2: 156,011 transactions


  Batch 3: 155,830 transactions


  Batch 4: 155,489 transactions


  Batch 5: 156,062 transactions


In [69]:
from pyspark.sql.types import StructType, StructField, \
    StringType, IntegerType, DoubleType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, \
    count, round as spark_round, col

stream_schema = StructType([
    StructField("Invoice",     StringType(),    True),
    StructField("StockCode",   StringType(),    True),
    StructField("Description", StringType(),    True),
    StructField("Quantity",    IntegerType(),   True),
    StructField("InvoiceDate", TimestampType(), True),
    StructField("Price",       DoubleType(),    True),
    StructField("CustomerID",  StringType(),    True),
    StructField("Country",     StringType(),    True)
])


# READ STREAM — monitor folder for new CSV files

stream_df = spark.readStream \
    .format("csv") \
    .schema(stream_schema) \
    .option("header", True) \
    .option("maxFilesPerTrigger", 1) \
    .load(stream_input_dir)

# Add revenue column
stream_with_revenue = stream_df.withColumn(
    "revenue", col("Quantity") * col("Price")
)

# STREAM 1 — Live revenue per country

stream_revenue = stream_with_revenue \
    .groupBy("Country") \
    .agg(
        spark_round(spark_sum("revenue"), 2).alias("total_revenue"),
        count("Invoice").alias("total_transactions")
    )

# STREAM 2 — High value transaction alerts (>£500)

stream_alerts = stream_with_revenue \
    .filter(col("revenue") > 500) \
    .select("CustomerID", "Description",
            "Quantity", "Country",
            spark_round("revenue", 2).alias("revenue_£"))

print("Stream queries defined!")
print("\nStream 1 — Revenue by Country (complete mode)")
print("Stream 2 — High Value Alerts >£500 (append mode)")

Stream queries defined!

Stream 1 — Revenue by Country (complete mode)
Stream 2 — High Value Alerts >£500 (append mode)


In [70]:
# ============================================================
# STOP ALL ACTIVE STREAMS — clean slate
# ============================================================
for q in spark.streams.active:
    q.stop()
    print(f"Stopped: {q.name}")

print(f"Active streams remaining: {len(spark.streams.active)}")

Active streams remaining: 0


In [71]:
# STREAM 1 — Revenue Dashboard

revenue_query = stream_revenue.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("live_revenue") \
    .option("checkpointLocation", "/tmp/checkpoint_revenue") \
    .trigger(processingTime="5 seconds") \
    .start()

print("Revenue stream started!")

# Write all batches
for i, batch in enumerate(batches):
    print(f"\nBatch {i+1}/5 arriving — {batch.count():,} transactions")
    batch.write \
        .mode("append") \
        .option("header", True) \
        .csv(stream_input_dir)
    time.sleep(6)

    print(f"📊 LIVE REVENUE — Batch {i+1}:")
    spark.sql("""
        SELECT Country, total_revenue, total_transactions
        FROM live_revenue
        ORDER BY total_revenue DESC
        LIMIT 5
    """).show(truncate=False)

revenue_query.stop()
print("✅ Stream stopped!")

26/06/04 16:44:10 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Revenue stream started!



Batch 1/5 arriving — 156,033 transactions


📊 LIVE REVENUE — Batch 1:
+-------+-------------+------------------+
|Country|total_revenue|total_transactions|
+-------+-------------+------------------+
+-------+-------------+------------------+



26/06/04 16:44:21 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 6749 milliseconds


...


Batch 5/5 arriving — 156,062 transactions


26/06/04 16:45:24 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 7016 milliseconds


📊 LIVE REVENUE — Batch 5:


+--------------+-------------+------------------+
|Country       |total_revenue|total_transactions|
+--------------+-------------+------------------+
|United Kingdom|8489962.52   |420221            |
|EIRE          |366640.99    |9326              |
|Netherlands   |334385.52    |3127              |
|Germany       |257064.24    |9885              |
|France        |204163.79    |8161              |
+--------------+-------------+------------------+

✅ Stream stopped!


26/06/04 16:45:29 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 12, writer: org.apache.spark.sql.execution.streaming.sources.MemoryStreamingWrite@1eccb494] is aborting.
26/06/04 16:45:29 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 12, writer: org.apache.spark.sql.execution.streaming.sources.MemoryStreamingWrite@1eccb494] aborted.


In [72]:
# STREAM 2 — High Value Alerts
import shutil

stream_input_dir2 = "/tmp/retail_stream_input2"
shutil.rmtree(stream_input_dir2, ignore_errors=True)
os.makedirs(stream_input_dir2, exist_ok=True)

# Stop any existing streams
for q in spark.streams.active:
    q.stop()

# Redefine stream on fresh directory
stream_df2 = spark.readStream \
    .format("csv") \
    .schema(stream_schema) \
    .option("header", True) \
    .option("maxFilesPerTrigger", 1) \
    .load(stream_input_dir2)

stream_alerts2 = stream_df2 \
    .withColumn("revenue", col("Quantity") * col("Price")) \
    .filter(col("revenue") > 500) \
    .select(
        "CustomerID", "Description", "Country",
        spark_round("revenue", 2).alias("revenue_£")
    )

alerts_query = stream_alerts2.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("live_alerts") \
    .option("checkpointLocation", "/tmp/checkpoint_alerts3") \
    .trigger(processingTime="5 seconds") \
    .start()

print("Alerts stream started!")

for i, batch in enumerate(batches):
    print(f"\nBatch {i+1}/5 arriving...")
    batch.write \
        .mode("append") \
        .option("header", True) \
        .csv(stream_input_dir2)
    
    # Wait longer for stream to process
    time.sleep(10)

    # Check if table has data before querying
    try:
        alert_count = spark.sql("SELECT COUNT(*) as n FROM live_alerts").first()["n"]
        if alert_count > 0:
            print(f"🚨 HIGH VALUE ALERTS — Batch {i+1} ({alert_count} total alerts):")
            spark.sql("""
                SELECT CustomerID, Description, Country, revenue_£
                FROM live_alerts
                ORDER BY revenue_£ DESC
                LIMIT 5
            """).show(truncate=False)
        else:
            print(f"⏳ Batch {i+1}: Stream processing... no alerts yet")
    except Exception as e:
        print(f"⏳ Batch {i+1}: Stream initialising...")

alerts_query.stop()
print("✅ Alerts stream stopped!")

26/06/04 16:45:29 WARN TaskSetManager: Lost task 199.0 in stage 926.0 (TID 3393) (cs-01kt9q92es8af28ybnw73qp03f.us-central1-c.c.lightning-ai-prod.internal executor driver): TaskKilled (Stage cancelled: Job 328 cancelled part of cancelled job group a90132cb-d204-4a0b-beb1-487e4754e1d0)
26/06/04 16:45:29 WARN TaskSetManager: Lost task 198.0 in stage 926.0 (TID 3392) (cs-01kt9q92es8af28ybnw73qp03f.us-central1-c.c.lightning-ai-prod.internal executor driver): TaskKilled (Stage cancelled: Job 328 cancelled part of cancelled job group a90132cb-d204-4a0b-beb1-487e4754e1d0)
26/06/04 16:45:29 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Alerts stream started!

Batch 1/5 arriving...


🚨 HIGH VALUE ALERTS — Batch 1 (187 total alerts):
⏳ Batch 1: Stream initialising...

Batch 2/5 arriving...


🚨 HIGH VALUE ALERTS — Batch 2 (369 total alerts):
⏳ Batch 2: Stream initialising...

Batch 3/5 arriving...


🚨 HIGH VALUE ALERTS — Batch 3 (546 total alerts):
⏳ Batch 3: Stream initialising...

Batch 4/5 arriving...


🚨 HIGH VALUE ALERTS — Batch 4 (818 total alerts):
⏳ Batch 4: Stream initialising...

Batch 5/5 arriving...


🚨 HIGH VALUE ALERTS — Batch 5 (988 total alerts):
⏳ Batch 5: Stream initialising...
✅ Alerts stream stopped!


In [73]:
# STREAMING SUMMARY — Final state after all batches

print("=" * 55)
print("FINAL STREAMING RESULTS — All 5 Batches Processed")
print("=" * 55)

print("\n📊 FINAL REVENUE BY COUNTRY (Top 10):")
spark.sql("""
    SELECT
        Country,
        total_revenue,
        total_transactions
    FROM live_revenue
    ORDER BY total_revenue DESC
    LIMIT 10
""").show(truncate=False)

print("\n🚨 TOTAL HIGH VALUE ALERTS DETECTED:")
spark.sql("""
    SELECT COUNT(*) as total_alerts,
           COUNT(DISTINCT CustomerID) as unique_customers
    FROM live_alerts
""").show()

print("\n💡 Streaming Insight:")
print("   Real-time processing enables instant marketing response:")
print("   → High value alerts trigger immediate VIP outreach")
print("   → Live revenue dashboard updates without batch delays")
print("   → Anomaly detection catches fraud as it happens")

FINAL STREAMING RESULTS — All 5 Batches Processed

📊 FINAL REVENUE BY COUNTRY (Top 10):
+--------------+-------------+------------------+
|Country       |total_revenue|total_transactions|
+--------------+-------------+------------------+
|United Kingdom|8489962.52   |420221            |
|EIRE          |366640.99    |9326              |
|Netherlands   |334385.52    |3127              |
|Germany       |257064.24    |9885              |
|France        |204163.79    |8161              |
|Australia     |101671.91    |1061              |
|Spain         |63561.6      |2192              |
|Switzerland   |58164.54     |1838              |
|Sweden        |56177.12     |810               |
|Denmark       |40018.86     |447               |
+--------------+-------------+------------------+


🚨 TOTAL HIGH VALUE ALERTS DETECTED:


+------------+----------------+
|total_alerts|unique_customers|
+------------+----------------+
|         988|             140|
+------------+----------------+


💡 Streaming Insight:
   Real-time processing enables instant marketing response:
   → High value alerts trigger immediate VIP outreach
   → Live revenue dashboard updates without batch delays
   → Anomaly detection catches fraud as it happens


In [74]:
# STREAMING TEST

# Stop any existing streams
for q in spark.streams.active:
    q.stop()

# Clean slate
import shutil
test_dir = "/tmp/stream_test"
shutil.rmtree(test_dir, ignore_errors=True)
os.makedirs(test_dir, exist_ok=True)

# Define fresh test stream
test_stream_df = spark.readStream \
    .format("csv") \
    .schema(stream_schema) \
    .option("header", True) \
    .option("maxFilesPerTrigger", 1) \
    .load(test_dir)

test_revenue = test_stream_df \
    .withColumn("revenue", col("Quantity") * col("Price")) \
    .groupBy("Country") \
    .agg(
        spark_round(spark_sum("revenue"), 2).alias("total_revenue"),
        count("Invoice").alias("transactions")
    )

test_query = test_revenue.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("test_stream") \
    .option("checkpointLocation", "/tmp/checkpoint_test") \
    .trigger(processingTime="3 seconds") \
    .start()

print("=" * 50)
print("STREAMING TEST — Watch numbers grow in real-time!")
print("=" * 50)

# Write ONE batch at a time with clear before/after
for i, batch in enumerate(batches[:3]):  # just 3 batches for test
    
    print(f"\n{'─'*50}")
    print(f"⏳ BEFORE Batch {i+1} — Current state:")
    try:
        before = spark.sql("SELECT COUNT(*) as countries, SUM(total_revenue) as revenue FROM test_stream").first()
        print(f"   Countries: {before['countries']} | Total Revenue: £{before['revenue'] or 0:,.2f}")
    except:
        print("   Stream empty — no data yet")

    # Write batch
    print(f"\n📦 Writing batch {i+1} — {batch.count():,} new transactions...")
    batch.write \
        .mode("append") \
        .option("header", True) \
        .csv(test_dir)
    
    # Wait for processing
    time.sleep(8)

    print(f"\n✅ AFTER Batch {i+1} — Updated state:")
    after = spark.sql("""
        SELECT Country, total_revenue, transactions
        FROM test_stream
        ORDER BY total_revenue DESC
        LIMIT 5
    """).show(truncate=False)

    # Show stream is still running
    print(f"   Stream status: {test_query.status['message']}")
    print(f"   Is active: {test_query.isActive}")

test_query.stop()
print("\n✅ Test complete — streaming proved working!")

STREAMING TEST — Watch numbers grow in real-time!

──────────────────────────────────────────────────
⏳ BEFORE Batch 1 — Current state:
   Countries: 0 | Total Revenue: £0.00


26/06/04 16:46:27 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.



📦 Writing batch 1 — 156,033 new transactions...


26/06/04 16:46:35 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 3000 milliseconds, but spent 5003 milliseconds



✅ AFTER Batch 1 — Updated state:


...

26/06/04 16:46:58 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 3000 milliseconds, but spent 4770 milliseconds



📦 Writing batch 3 — 155,830 new transactions...


26/06/04 16:47:03 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 3000 milliseconds, but spent 5005 milliseconds



✅ AFTER Batch 3 — Updated state:
+--------------+-------------+------------+
|Country       |total_revenue|transactions|
+--------------+-------------+------------+
|United Kingdom|5624171.91   |280229      |
|EIRE          |250951.38    |6296        |
|Netherlands   |223035.31    |2089        |
|Germany       |170933.12    |6627        |
|France        |137070.98    |5443        |
+--------------+-------------+------------+

   Stream status: Processing new data
   Is active: True

✅ Test complete — streaming proved working!


26/06/04 16:47:07 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 3000 milliseconds, but spent 4229 milliseconds
